# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
FAIR^2 dataset, containing ordered logistic regression results and survey data on indigenous and modern knowledge adoption in rangeland management, is provided as a Croissant schema JSON-LD.

- URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
- License: [ODC-BY 1.0](https://opendatacommons.org/licenses/by/1-0/)


In [ ]:
# Install mlcroissant if it's not already present
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
# Define the dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset via the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
List all available record sets in the dataset, with their `@id`s. For each record set, enumerate its fields and columns (by `@id`).

In [ ]:
# Show available record sets, and explore each's fields by @id

record_sets = list(dataset.record_sets)
if record_sets:
    print(f"Available record sets ({len(record_sets)}):\n")
    for rs in record_sets:
        print(f"- Record set: {rs['@id']}")
        # If fields or columns are present, list them by @id
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  Fields by @id:")
            for f in fields:
                if isinstance(f, dict):
                    print(f"    - {f.get('@id', f)}")
                else:
                    print(f"    - {f}")
        if 'column' in rs:
            columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
            print("  Columns by @id:")
            for c in columns:
                if isinstance(c, dict):
                    print(f"    - {c.get('@id', c)}")
                else:
                    print(f"    - {c}")
        print()
else:
    print("No record sets are defined in the schema or accessible.")

## 3. Data Extraction
Let's load the records from each record set into pandas DataFrames. All entities (record sets, fields, columns) are referenced by their respective `@id`s.

In [ ]:
# Extract all record sets by their @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # mlcroissant expects the 'record_set' argument to be the @id
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set: {record_set_id}")
        else:
            print(f"Record set {record_set_id} is empty or not materialized.")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")
        continue

if dataframes:
    # Pick the first DataFrame for demo
    first_rs = list(dataframes.keys())[0]
    print(f"\nFirst record set columns (@id): {list(dataframes[first_rs].columns)}\n")
    display(dataframes[first_rs].head())
else:
    print("No DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
We will perform some exploratory analysis on the first available record set. This includes filtering, normalization, and grouping, always referencing columns by their `@id`s.

In [ ]:
# EDA on the first loaded record set, using field @id

if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    print(f"Working with record set: {df_key}")
    print(f"Available columns (@id):\n{list(df.columns)}\n")

    # Try to automatically select a numeric field by inferring dtypes
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) == 0:
        print("No numeric field identified in this record set.")
    else:
        # Use the first numeric @id (column name)
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field for analysis: {numeric_field_id}")

        # Filter: keep records where numeric field exceeds threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows\n")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
        )

        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(
            filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head()
        )

        # Try to find a categorical column for grouping
        object_cols = df.select_dtypes(include=['object']).columns
        group_field_id = None
        for col in object_cols:
            # Prefer a field with few unique values
            if 2 <= df[col].nunique() <= 12:
                group_field_id = col
                break

        if group_field_id:
            print(f"\nGrouping by field: {group_field_id}")
            # Group and summarize the normalized field
            grouped_df = (
                filtered_df.groupby(group_field_id)[f"{numeric_field_id}_normalized"].mean().reset_index()
            )
            display(grouped_df.head())
        else:
            print("No suitable categorical/grouping field found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the (normalized) numeric field distribution and relationship with one grouping/categorical field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[f"{numeric_field_id}_normalized"])
        plt.title(f"Normalized {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"{numeric_field_id}_normalized")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore a Croissant-based dataset using the `mlcroissant` library. We loaded the FAIR^2 dataset, listed its available record sets and fields by `@id`, extracted records into dataframes, and performed initial exploration, including filtering, normalization, and basic visualization.

**Key learnings:**
- How to access and process data using the Croissant schema standard and the `mlcroissant` Python tools
- The importance of referencing all dataset elements by their `@id`
- Dataset contains valuable information on knowledge adoption in Northern Kenya, suitable for policy, research, and analytical purposes. Further domain-specific exploration can yield actionable insights for inclusive management strategies.
